# Barrier Options and Path Dependency: A Student Monte Carlo Study

**Project type:** semi-learning paper notebook  
**Research question:** why do some options require modelling the whole price path rather than only the terminal price?

## Abstract

This note introduces barrier options as a simple example of path-dependent derivatives. A vanilla European call depends only on the terminal stock price. A barrier option also depends on whether the underlying touched a pre-defined level before expiry. I use Monte Carlo simulation to price down-and-out calls and compare them with vanilla Black-Scholes prices. The purpose is to understand how path dependency changes the pricing problem and why simulation becomes useful when the payoff depends on the journey, not only the destination.

## Personal Learning Position

Barrier options are interesting to me because they make a weakness of the basic Black-Scholes formula obvious. In the vanilla formula, the terminal distribution is enough. For a barrier option, knowing the final price is not sufficient. A stock can end above the strike but still have knocked out earlier. This feels like a natural step from textbook option pricing toward structured products and more realistic payoff design.

In [ ]:
from pathlib import Path
import sys

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "src").exists():
        sys.path.insert(0, str(candidate / "src"))
        PROJECT_ROOT = candidate
        break

import matplotlib.pyplot as plt
import pandas as pd

from options_pricing_research.black_scholes import black_scholes_price
from options_pricing_research.monte_carlo import barrier_discount_to_vanilla, barrier_monte_carlo_price

plt.style.use("seaborn-v0_8-whitegrid")
pd.options.display.float_format = "{:.4f}".format

## 1. Payoff Intuition

A down-and-out call behaves like a call option unless the underlying price falls to or below a barrier level. If the barrier is touched, the option expires worthless. This creates a discount relative to the vanilla call because some paths that would have positive terminal payoff are removed.

In [ ]:
spot = 100.0
strike = 100.0
time_to_expiry = 1.0
risk_free_rate = 0.03
volatility = 0.25

vanilla_call = black_scholes_price("call", spot, strike, time_to_expiry, risk_free_rate, volatility)
vanilla_call

## 2. Barrier Level Sensitivity

A lower down-and-out barrier is less likely to be touched, so its price should be closer to the vanilla call. A higher barrier is more restrictive and should make the option cheaper.

In [ ]:
barriers = [70.0, 80.0, 85.0, 90.0, 95.0]
barrier_table = pd.DataFrame(
    [
        {
            "barrier": barrier,
            **barrier_discount_to_vanilla(
                "call",
                "down-and-out",
                spot,
                strike,
                barrier,
                time_to_expiry,
                risk_free_rate,
                volatility,
                paths=25_000,
                steps=126,
                seed=int(barrier),
            ),
        }
        for barrier in barriers
    ]
)
barrier_table

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(barrier_table["barrier"], barrier_table["barrier_price"], marker="o", label="down-and-out call")
ax.axhline(vanilla_call, color="black", linestyle="--", label="vanilla call")
ax.set_title("Barrier Level and Knock-Out Option Price")
ax.set_xlabel("Barrier level")
ax.set_ylabel("Option price")
ax.legend()
plt.show()

My takeaway is that the barrier is not a small contract detail. It can change the option value materially because it changes which paths survive. This helps me understand why structured products can have payoffs that sound simple in words but require careful modelling in code.

## 3. Knock-Out Logic

As a basic sanity check, if the spot starts below a down-and-out barrier, the option should already be knocked out and worth zero.

In [ ]:
already_knocked_out = barrier_monte_carlo_price(
    "call",
    "down-and-out",
    spot=80.0,
    strike=100.0,
    barrier=85.0,
    time_to_expiry=1.0,
    risk_free_rate=0.03,
    volatility=0.25,
)
already_knocked_out

This is a simple check, but I think it matters. In quantitative finance, a model can produce impressive charts and still be wrong if the edge cases are not handled. For path-dependent products, contract logic is part of the model.

## 4. Reflection

Barrier options made me appreciate why Monte Carlo is more than just a numerical trick. For vanilla options, simulation is slower than Black-Scholes. For path-dependent options, simulation becomes a natural language for expressing the payoff. My current view is that the main challenge is not only mathematical pricing, but also translating the legal/economic payoff description into accurate program logic.

## References

- Hull, J. C. *Options, Futures, and Other Derivatives*.
- Glasserman, P. *Monte Carlo Methods in Financial Engineering*.